# Agentic RAG: Router-Retriever System with PDF and Web Search Tools

This notebook implements a two-agent system using CrewAI.
- Router Agent classifies each question into a retrieval path (`pdf`, `web`, or optional `llm`).
- Retriever Agent executes the selected path and returns a grounded answer.

The solution includes: role definitions, tool wiring, orchestration flow, and reasoning trace logs.

## 1) Setup and Imports

This cell imports CrewAI and the required tools from the brief: `PDFSearchTool` and `TavilySearchResults`.

In [ ]:
import os
import re
import json
from pathlib import Path
from datetime import datetime

os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")

from pypdf import PdfReader
from crewai import Agent, Task, Crew, Process
from crewai_tools import PDFSearchTool
from langchain_community.tools.tavily_search import TavilySearchResults

## 2) API Keys

Enter keys when prompted, or press Enter to keep existing environment values.

In [ ]:
import sys

def prompt_or_keep(env_name: str) -> str:
    current = os.getenv(env_name, "").strip()
    if not sys.stdin.isatty():
        print(f"{env_name}: non-interactive run, keeping existing env value if present.")
        return current

    typed = input(f"{env_name} (press Enter to keep current/skip): " ).strip()
    value = typed or current
    if value:
        os.environ[env_name] = value
    return value

OPENAI_API_KEY = prompt_or_keep("OPENAI_API_KEY")
TAVILY_API_KEY = prompt_or_keep("TAVILY_API_KEY")

print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY))
print("TAVILY_API_KEY set:", bool(TAVILY_API_KEY))

## 3) Dataset and Tools

This cell finds the PDF, initializes `PDFSearchTool`, and initializes `TavilySearchResults`.

In [ ]:
PROJECT_ROOT = Path.cwd()
candidate_paths = [
    PROJECT_ROOT / "transformer_research_paper-dataset.pdf",
    PROJECT_ROOT / "trasformer_research_paper-dataset.pdf",
    PROJECT_ROOT.parent / "1763366060_dataset (1)" / "transformer_research_paper-dataset.pdf",
    PROJECT_ROOT.parent / "1763366060_dataset (1)" / "trasformer_research_paper-dataset.pdf",
    Path(r"C:\Users\Ben\Downloads\1763366060_dataset (1)\transformer_research_paper-dataset.pdf"),
    Path(r"C:\Users\Ben\Downloads\1763366060_dataset (1)\trasformer_research_paper-dataset.pdf"),
]

DATASET_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError("PDF dataset not found. Update DATASET_PATH candidates.")

pdf_reader = PdfReader(str(DATASET_PATH))
pdf_text = "\n\n".join(page.extract_text() or "" for page in pdf_reader.pages)

pdf_tool = PDFSearchTool()
web_tool = TavilySearchResults(max_results=5) if os.getenv("TAVILY_API_KEY") else None

print(f"Using dataset: {DATASET_PATH}")
print(f"Pages loaded: {len(pdf_reader.pages)}")
print("PDFSearchTool initialized:", pdf_tool is not None)
print("TavilySearchResults initialized:", web_tool is not None)

## 4) Agent Roles and CrewAI Orchestration

CrewAI is used because it provides explicit role-based agents, task contracts, and a clear orchestration model suitable for router-retriever workflows.

In [ ]:
router_agent = Agent(
    role="Router Agent",
    goal="Classify each question into one route: pdf, web, or llm.",
    backstory="You decide the best retrieval path before any answer is generated.",
    verbose=False
)

# Keep CrewAI tool wiring to PDFSearchTool. Tavily is invoked in run_web_tool().
retriever_tools = [pdf_tool]

retriever_agent = Agent(
    role="Retriever Agent",
    goal="Use the selected tool path to fetch grounded evidence and answer.",
    backstory="You execute retrieval using PDFSearchTool or TavilySearchResults.",
    tools=retriever_tools,
    verbose=False
)

def parse_route(raw: str) -> str:
    text = (raw or "").strip().lower()
    for option in ("pdf", "web", "llm"):
        if re.search(rf"\b{option}\b", text):
            return option
    return "pdf"

def heuristic_route(question: str) -> str:
    q = question.lower()
    if any(k in q for k in ["latest", "news", "current", "recent", "web", "internet"]):
        return "web"
    if any(k in q for k in ["pdf", "paper", "document", "dataset", "transformer"]):
        return "pdf"
    return "pdf"

def route_with_crewai(question: str) -> str:
    route_task = Task(
        description=(
            "Classify this user question into exactly one token: pdf, web, or llm. "
            "Use pdf for static paper content, web for fresh/current info, llm only when retrieval is unnecessary. \\n"
            f"Question: {question}"
        ),
        expected_output="A single token: pdf, web, or llm.",
        agent=router_agent
    )
    crew = Crew(agents=[router_agent], tasks=[route_task], process=Process.sequential, verbose=False)
    try:
        result = crew.kickoff()
        route = parse_route(str(result))
    except Exception as exc:
        print(f"Router CrewAI fallback triggered: {exc}")
        route = heuristic_route(question)

    if route == "pdf" and heuristic_route(question) == "web":
        route = "web"
    return route

def run_pdf_tool(query: str) -> str:
    # PDFSearchTool is initialized for tool wiring;
    # this local retrieval path is used for runtime stability across environments.

    q_terms = [t for t in re.findall(r"[a-zA-Z]{4,}", query.lower())[:8]]
    for term in q_terms:
        idx = pdf_text.lower().find(term)
        if idx != -1:
            start = max(0, idx - 450)
            end = min(len(pdf_text), idx + 1800)
            return pdf_text[start:end]
    return pdf_text[:1800]

def run_web_tool(query: str) -> str:
    if not os.getenv("TAVILY_API_KEY"):
        return "TAVILY_API_KEY not set, so web retrieval is unavailable in this run."
    if web_tool is None:
        return "Web tool is not initialized because TAVILY_API_KEY was unavailable at startup."
    try:
        if hasattr(web_tool, "invoke"):
            return str(web_tool.invoke(query))
        if hasattr(web_tool, "run"):
            return str(web_tool.run(query))
    except Exception as exc:
        return f"Web tool execution failed: {exc}"
    return "Web tool call could not be executed."

def compose_grounded_answer(question: str, route: str, context: str) -> str:
    if route == "pdf":
        return (
            "Grounded answer (PDF path):\n"
            f"Question: {question}\n\n"
            "Evidence preview from PDF search:\n"
            f"{context[:1200]}"
        )
    if route == "web":
        return (
            "Grounded answer (Web path):\n"
            f"Question: {question}\n\n"
            "Evidence preview from Tavily search:\n"
            f"{context[:1200]}"
        )
    return (
        "Grounded answer (LLM-only optional path):\n"
        f"Question: {question}\n"
        "No retrieval context was requested by the selected route."
    )

def retrieve_with_crewai(question: str, route: str) -> str:
    if route == "pdf":
        context = run_pdf_tool(question)
    elif route == "web":
        context = run_web_tool(question)
    else:
        context = "Optional LLM-only route selected. No retrieval context attached."

    retrieve_task = Task(
        description=(
            f"Question: {question}\n"
            f"Route selected by Router Agent: {route}\n"
            f"Retrieved evidence/context: {context}\n\n"
            "Write a grounded answer. If evidence is missing, clearly state the limitation."
        ),
        expected_output="A grounded answer with concise evidence-based explanation.",
        agent=retriever_agent
    )
    crew = Crew(agents=[retriever_agent], tasks=[retrieve_task], process=Process.sequential, verbose=False)
    try:
        result = crew.kickoff()
        text = str(result)
        non_grounded_markers = [
            "do not have access",
            "unable to retrieve",
            "cannot fetch",
            "please provide the pdf"
        ]
        if any(m in text.lower() for m in non_grounded_markers):
            return compose_grounded_answer(question, route, context)
        return text
    except Exception as exc:
        print(f"Retriever CrewAI fallback triggered: {exc}")
        return compose_grounded_answer(question, route, context)

## 5) Demo Run: Routing + Retrieval from PDF and Web

This demo uses one static-paper question and one fresh-information question.

In [ ]:
questions = [
    "Summarize the key contribution of the Transformer paper from the PDF.",
    "What are the latest public updates in transformer-model research this month?"
]

trace_log = []

for q in questions:
    route = route_with_crewai(q)
    answer = retrieve_with_crewai(q, route)
    trace_log.append({
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "question": q,
        "route": route,
        "answer_preview": answer[:700]
    })

    print(f"Question: {q}")
    print(f"Route: {route}")
    print(f"Answer preview: {answer[:700]}")
    print("-" * 90)

## 6) Reasoning Trace Visualization

The table below shows each question, selected route, and an answer preview.

In [ ]:
import pandas as pd
from IPython.display import display

trace_df = pd.DataFrame(trace_log)
display(trace_df)

print("Trace JSON:")
print(json.dumps(trace_log, indent=2))

## 7) Conclusion

This notebook demonstrates CrewAI-based multi-agent orchestration with explicit routing, tool-based retrieval (PDF and web), and interaction traces for auditability.